In [ ]:
import os
import ast
import json
import torch
import pickle
import numpy as np
from tqdm import tqdm
from collections import defaultdict, Counter, OrderedDict
from pathlib import Path
from datasets import load_dataset, Dataset
import pandas as pd
import random
import collections
from scipy.stats import pearsonr
from pathlib import Path
from typing import Dict, List, Sequence
import copy
from typing import Dict, Any, List
random.seed(9001)

In [ ]:
import mteb
from mteb.task_selection import results_to_dataframe
from mteb.abstasks.AbsTask import AbsTask
from mteb.load_results.mteb_results import MTEBResults
from mteb.model_meta import ModelMeta

### Environmental Variables

In [ ]:
RESULT_DIR = 
DATASET_DIR = 

In [ ]:
def load_results(result_dir: str,
                    models: List[str],
                    tasks: List[str],
                    score_name: str='main_score') -> Dict[str, Dict[str, Any]]:
    """
    Load PIR results and compute the mean score across tasks for each model.

    Args:
        result_dir (str): Directory containing the results.
        models (List[str]): List of model names.
        tasks (List[str]): List of task names.

    Returns:
        Dict[str, Dict[str, Any]]: Nested dictionary with model names as keys,
            each containing a dictionary of task scores and a 'mean' key for the average score.
    """
    
    results = defaultdict(dict)
    for _model in models:
        _model_path = os.path.join(result_dir, _model.replace('/', '__'))
        assert os.path.isdir(_model_path), f"Model {_model} not found in {result_dir}"
        
        # Only one directory under _model_path
        directories = [d for d in os.listdir(_model_path) if os.path.isdir(os.path.join(_model_path, d))]
        assert len(directories) == 1, f"Model {_model} has more than one directory"
        
        sub_directories = [
            d for d in os.listdir(os.path.join(_model_path, directories[0]))
            if os.path.isdir(os.path.join(_model_path, directories[0], d))
        ]
        assert len(sub_directories) == 1, f"Model {_model} has more than one sub-directory"
        
        scores = []  # List to store scores for computing the mean
        for _task in tasks:
            _task_path = os.path.join(
                _model_path, directories[0], sub_directories[0], f'{_task}.json'
            )
            if os.path.isfile(_task_path):
                with open(_task_path, 'r') as f:
                    score = np.mean([_score[score_name] for _score in json.load(f)['scores']['test']])
                    results[_model][_task] = score
                    scores.append(score)
            else:
                print(f"Task {_task} not found in {result_dir}/{_model}")
                results[_model][_task] = 0
                scores.append(0)
        
        # Compute and store the mean score for the current model
        if scores:
            mean_score = sum(scores) / len(scores)
            results[_model]['mean'] = mean_score
        else:
            results[_model]['mean'] = None  # Handle case with no scores

    return results

def highlight_max_in_column(s):
    """
    Highlights the maximum value in a Series with bold text.

    Args:
        s (pd.Series): A pandas Series representing a column in the DataFrame.

    Returns:
        List[str]: A list of styles for each element in the Series.
    """
    is_max = s == s.max()
    return ['font-weight: bold' if v else '' for v in is_max]

In [ ]:
def load_finegrained_results(result_dir: str,
                    models: List[str],
                    tasks: List[str],
                    score_name: str = 'main_score') -> Dict[str, Dict[str, Any]]:
    """
    Load PIR results and compute the mean score across tasks for each model.

    Args:
        result_dir (str): Directory containing the results.
        models (List[str]): List of model names.
        tasks (List[str]): List of task names.

    Returns:
        Dict[str, Dict[str, Any]]: Nested dictionary with model names as keys,
            each containing a dictionary of task scores and a 'mean' key for the average score.
    """
    
    results = defaultdict(dict)
    for _model in models:
        _model_path = os.path.join(result_dir, _model.replace('/', '__'))
        assert os.path.isdir(_model_path), f"Model {_model} not found in {result_dir}"
        
        # Only one directory under _model_path
        directories = [d for d in os.listdir(_model_path) if os.path.isdir(os.path.join(_model_path, d))]
        assert len(directories) == 1, f"Model {_model} has more than one directory"
        
        sub_directories = [
            d for d in os.listdir(os.path.join(_model_path, directories[0]))
            if os.path.isdir(os.path.join(_model_path, directories[0], d))
        ]
        assert len(sub_directories) == 1, f"Model {_model} has more than one sub-directory"
        
        scores = []  # List to store scores for computing the mean
        for _task in tasks:
            _task_path = os.path.join(
                _model_path, directories[0], sub_directories[0], f'{_task}.json'
            )
            if os.path.isfile(_task_path):
                with open(_task_path, 'r') as f:
                    for _hf_subset, _score in [(_score['hf_subset'], _score[score_name]) for _score in json.load(f)['scores']['test']]:
                        results[_model][_task + '-' + _hf_subset] = _score
                        scores.append(_score)
            else:
                print(f"Task {_task} not found in {result_dir}/{_model}")
                results[_model][_task] = 0
                scores.append(0)
        
        # Compute and store the mean score for the current model
        if scores:
            mean_score = sum(scores) / len(scores)
            results[_model]['mean'] = mean_score
        else:
            results[_model]['mean'] = None  # Handle case with no scores

    return results

## 1. Stardard Code Retrieval

In [ ]:
tasks = [
    'CodeNetBugPreferenceRetrieval',
    'CodeNetEfficiencyPreferenceRetrieval',
    'CVEFixesPreferenceRetrieval',
    'Defects4JPreferenceRetrieval',
    'DeprecatedCodePreferenceRetrieval',
    'SaferCodePreferenceRetrieval',
    'SQLR2PreferenceRetrieval',
]

models = [
    'bm25s',
    'facebook__contriever',
    'Alibaba-NLP__gte-base-en-v1.5',
    'sentence-transformers__gtr-t5-base',
    'sentence-transformers__gtr-t5-large',
    'intfloat__e5-base-v2',
    'intfloat__e5-large-v2',
    'Alibaba-NLP__gte-Qwen2-1.5B-instruct',
    'intfloat__e5-mistral-7b-instruct',
    'hkunlp__instructor-base',
    'hkunlp__instructor-large',
    'hkunlp__instructor-xl',
    'samaya-ai__promptriever-llama2-7b-v1',
    'samaya-ai__promptriever-llama3.1-8b-v1',
    'samaya-ai__promptriever-llama3.1-8b-instruct-v1',
    'samaya-ai__promptriever-mistral-v0.1-7b-v1',
    'openai__text-embedding-ada-002',
    'openai__text-embedding-3-small',
    'openai__text-embedding-3-large',
    'voyageai__voyage-code-2',
    'voyageai__voyage-code-3',
    'microsoft__codebert-base',
    'microsoft__graphcodebert-base',
    'nomic-ai__CodeRankEmbed',
    'local-repllama-llama31-8b-lora-64',
    'local-repllama-llama31-8b-lora-64-quality',
    'local-repllama-llama32-3b-lora-256',
    'local-repllama-llama32-3b-lora-256-quality',
    'codesage__codesage-small',
    'codesage__codesage-base',
]

### nDCG@10

In [ ]:
data = load_results(RESULT_DIR, models, tasks)
df = pd.DataFrame.from_dict(data, orient='index').style.apply(highlight_max_in_column, axis=0)

df

### MRR@10

In [ ]:
data = load_results(RESULT_DIR, models, tasks, 'mrr_at_10')
df = pd.DataFrame.from_dict(data, orient='index').style.apply(highlight_max_in_column, axis=0)

df

## 2. Preference-guided Code Retrieval

#### Load golden labels

In [ ]:
def load_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():  # skip empty lines
                data.append(json.loads(line))
    return data

In [ ]:
# the map between tasks an subtasks
task_sub_task_mapping = {
    "CodeNetBugPreferenceRetrieval": ["c","cpp", "go", "java", "javascript", "python", "ruby", "rust", "swift", "typescript"],
    "CodeNetEfficiencyPreferenceRetrieval": ["c","cpp", "go", "java", "javascript", "python", "ruby", "rust", "swift", "typescript"],
    "CVEFixesPreferenceRetrieval": ['c', 'go', 'java', 'python', 'ruby'],
    "Defects4JPreferenceRetrieval": [None],
    "DeprecatedCodePreferenceRetrieval": ['numpy', 'pandas', 'pytorch', 'scipy', 'seaborn', 'sklearn', 'tensorflow', 'transformers'],
    "SaferCodePreferenceRetrieval": ["c", "cpp", "python", "java", "javascript", "go", "ruby"],
    "SQLR2PreferenceRetrieval": [None],
}

In [ ]:
# load the golden qrels
datadir_mapping = {
    "CodeNetBugPreferenceRetrieval": "codenet_bug",
    "CodeNetEfficiencyPreferenceRetrieval": "codenet_effi",
    "CVEFixesPreferenceRetrieval": "CVEFixes",
    "Defects4JPreferenceRetrieval": "Defects4J",
    "DeprecatedCodePreferenceRetrieval": "DeprecatedCode",
    "SaferCodePreferenceRetrieval": "SafeCoder",
    'SQLR2PreferenceRetrieval': "sqlr2",
}

In [ ]:
def get_lang(_doc_str):
    return _doc_str.split("-")[1]

def get_lang_codenet(_doc_str):
    return _doc_str.split("-")[-2].lower()

def load_qrels(task, lang):
    if task=='DeprecatedCodePreferenceRetrieval':
        qrels_path = os.path.join(DATASET_DIR, datadir_mapping[task], f'qrels-{lang}.jsonl')
    else:
        qrels_path = os.path.join(DATASET_DIR, datadir_mapping[task], 'qrels.jsonl')

    lines = load_jsonl(qrels_path)

    # filtering only for CVEFixesPreferenceRetrieval
    if task == 'CVEFixesPreferenceRetrieval' or task == 'SaferCodePreferenceRetrieval':
        tmp_lines = []
        for _line in lines:
            filtered_pos_docids = [_id for _id in _line['pos-docids'] if get_lang(_id) == lang]
            filtered_neg_docids = [_id for _id in _line['neg-docids'] if get_lang(_id) == lang]
            _line['pos-docids'] = filtered_pos_docids
            _line['neg-docids'] = filtered_neg_docids
            if len(filtered_neg_docids) != 0 and len(filtered_neg_docids) != 0:
                tmp_lines.append(_line)

        lines = tmp_lines

    if task.startswith("CodeNet"):
        tmp_lines = []
        for _line in lines:
            filtered_pos_docids = [_id for _id in _line['pos-docids'] if get_lang_codenet(_id) == lang]
            filtered_neg_docids = [_id for _id in _line['neg-docids'] if get_lang_codenet(_id) == lang]
            _line['pos-docids'] = filtered_pos_docids
            _line['neg-docids'] = filtered_neg_docids
            if len(filtered_neg_docids) != 0 and len(filtered_neg_docids) != 0:
                tmp_lines.append(_line)

        lines = tmp_lines

    qrels_dict = {_line['qid']: _line for _line in lines}
    return qrels_dict

#### Obtain the prediction

In [ ]:
def load_predictions(model, task, subtask):
    prediction_file = f'{task}_{subtask}_predictions.json' if subtask is not None else f'{task}_default_predictions.json'
    prediction_path = os.path.join(result_dir, model, prediction_file)

    with open(prediction_path, 'r') as f:
        json_dict = json.loads(f.read())

    json_sorted_dict = {}
    for _k, _v in json_dict.items():
        sorted_doc_values = sorted([(_docid, _score) for _docid, _score in _v.items()], key=lambda x: x[1], reverse=True)
        json_sorted_dict[_k] = [_item[0] for _item in sorted_doc_values]

    return json_sorted_dict

## Metrics for Quality-aware Code Information Retrieval

Assume for each query, we have:
- A set of positive samples: $\mathcal{P} = \{p_1, p_2, \dots, p_m\}$
- A set of negative samples: $\mathcal{N} = \{n_1, n_2, \dots, n_n\}$
- Each sample has a model score: $s(x)$
- Total number of candidates: $C = |\mathcal{P}| + |\mathcal{N}|$

### 2.1: Pairwise Preference Accuracy (PPA)

Measures the proportion of positive-negative pairs where the positive is scored higher:

$$
\text{PPA} = \frac{1}{|\mathcal{P}| \cdot |\mathcal{N}|} \sum_{p \in \mathcal{P}} \sum_{n \in \mathcal{N}} \mathbb{1}(s(p) > s(n))
$$

- **Upper bound (best case)**: $\text{PPA} = 1$, meaning all positives are scored above all negatives.
- **Lower bound (worst case)**: $\text{PPA} = 0$, meaning all negatives are scored above all positives.

In [ ]:
def compute_ppa(_qrels_dict, _predictions):
    overall_ppa = []
    for _qrels_key in list(_qrels_dict.keys()):
        pos_docids = _qrels_dict[_qrels_key]['pos-docids']
        neg_docids = _qrels_dict[_qrels_key]['neg-docids']

        query_ppa = []
        for _pos_docid in pos_docids:
            _pos_docid_rank = _predictions[_qrels_key].index(_pos_docid) if _pos_docid in _predictions[_qrels_key] else len(_predictions[_qrels_key])
            for _neg_docid in neg_docids:
                _neg_docid_rank = _predictions[_qrels_key].index(_neg_docid) if _neg_docid in _predictions[_qrels_key] else len(_predictions[_qrels_key])
                if _pos_docid_rank < _neg_docid_rank:
                    query_ppa.append(1)
                else:
                    query_ppa.append(0)
        overall_ppa.append(np.mean(query_ppa))

    assert len(overall_ppa) > 0
    return np.mean(overall_ppa)

In [ ]:
ppa_dict = defaultdict(dict)

for _model in tqdm(models):
    for _task in tasks:
        ppa_values = []
        for _subtask in task_sub_task_mapping[_task]:
            _qrels_dict = load_qrels(_task, _subtask)
            _predictions = load_predictions(_model, _task, _subtask)
            _ppa_value = compute_ppa(_qrels_dict, _predictions)
            ppa_values.append(_ppa_value)
            ppa_dict[_model][_task + '-' + _subtask if _subtask is not None else _task] = _ppa_value
        
        # Compute mean PPA over subtasks
        if _task not in ppa_dict[_model]:
            mean_ppa = sum(ppa_values) / len(ppa_values) if ppa_values else None
            ppa_dict[_model][_task] = mean_ppa

df = pd.DataFrame.from_dict(ppa_dict, orient='index').style.apply(highlight_max_in_column, axis=0)

In [ ]:
# original_df is already your DataFrame
ppa_df = pd.DataFrame.from_dict(ppa_dict, orient='index')
mask = ~ppa_df.columns.astype(str).str.contains(r'-')
ppa_mean_df = ppa_df.loc[:, mask]

# inspect the result
ppa_mean_df

### 2.2: Margin-based Ranking Score (MRS)

Measures the average margin between positive and negative scores - in order to normalize the score across different datasets and retrievers, we use rank reciprocal function $r$

$$
\text{MRS} = \frac{1}{|\mathcal{P}| \cdot |\mathcal{N}|} \sum_{p \in \mathcal{P}} \sum_{n \in \mathcal{N}} \left( r(p) - r(n) \right)
$$

- **Upper bound**: $\text{MRS} \to 1$ if positives are scored much higher than negatives.
- **Lower bound**: $\text{MRS} \to -1$ if negatives are scored much higher than positives.

In [ ]:
def compute_mrs(_qrels_dict, _predictions):
    overall_mrs = []
    for _qrels_key in list(_qrels_dict.keys()):
        pos_docids = _qrels_dict[_qrels_key]['pos-docids']
        neg_docids = _qrels_dict[_qrels_key]['neg-docids']
        query_mrs = []
        for _pos_docid in pos_docids:
            _pos_docid_rank = _predictions[_qrels_key].index(_pos_docid) if _pos_docid in _predictions[_qrels_key] else len(_predictions[_qrels_key])
            for _neg_docid in neg_docids:
                _neg_docid_rank = _predictions[_qrels_key].index(_neg_docid) if _neg_docid in _predictions[_qrels_key] else len(_predictions[_qrels_key])
                query_mrs.append(1./(1+_pos_docid_rank) - 1./(1+_neg_docid_rank))
        overall_mrs.append(np.mean(query_mrs))
    return np.mean(overall_mrs)

In [ ]:
mrs_dict = defaultdict(dict)
for _model in tqdm(models):
    mrs_values = []
    for _task in tasks:
        for _subtask in task_sub_task_mapping[_task]:
            _qrels_dict = load_qrels(_task, _subtask)
            _predictions = load_predictions(_model, _task, _subtask)
            _mrs_value = compute_mrs(_qrels_dict, _predictions)
            mrs_values.append(_mrs_value)
            mrs_dict[_model][_task + '-' + _subtask if _subtask is not None else _task] = _mrs_value
        
        # Compute mean MRS over subtasks
        if _task not in mrs_dict[_model]:
            mean_mrs = sum(mrs_values) / len(mrs_values) if mrs_values else None
            mrs_dict[_model][_task] = mean_mrs

df = pd.DataFrame.from_dict(mrs_dict, orient='index').style.apply(highlight_max_in_column, axis=0)

In [ ]:
mrs_df = pd.DataFrame.from_dict(mrs_dict, orient='index')
# original_df is already your DataFrame
mask = ~mrs_df.columns.astype(str).str.contains(r'-')
mrs_mean_df = mrs_df.loc[:, mask]

# inspect the result
mrs_mean_df

### 3. Analysis

#### 3.1 PPA across multiple languages

In [ ]:
ppa_df

In [ ]:
mrs_df

#### 3.2 Instruction Ablation

In [ ]:
ablation_models = [
    'hkunlp__instructor-base_wo_instruction',
    'hkunlp__instructor-large_wo_instruction',
    'hkunlp__instructor-xl_wo_instruction',
    'samaya-ai__promptriever-llama2-7b-v1_wo_instruction',
    'samaya-ai__promptriever-llama3.1-8b-v1_wo_instruction',
    'samaya-ai__promptriever-llama3.1-8b-instruct-v1_wo_instruction',
    'samaya-ai__promptriever-mistral-v0.1-7b-v1_wo_instruction',
    'hkunlp__instructor-base_neg_instruction',
    'hkunlp__instructor-large_neg_instruction',
    'hkunlp__instructor-xl_neg_instruction',
    'samaya-ai__promptriever-llama2-7b-v1_neg_instruction',
    'samaya-ai__promptriever-llama3.1-8b-v1_neg_instruction',
    'samaya-ai__promptriever-llama3.1-8b-instruct-v1_neg_instruction',
    'samaya-ai__promptriever-mistral-v0.1-7b-v1_neg_instruction'
]

##### PPA

In [ ]:
ppa_dict = defaultdict(dict)

for _model in tqdm(ablation_models):
    for _task in tasks:
        ppa_values = []
        for _subtask in task_sub_task_mapping[_task]:
            _qrels_dict = load_qrels(_task, _subtask)
            _predictions = load_predictions(_model, _task, _subtask)
            _ppa_value = compute_ppa(_qrels_dict, _predictions)
            ppa_values.append(_ppa_value)
            ppa_dict[_model][_task + '-' + _subtask if _subtask is not None else _task] = _ppa_value
        
        # Compute mean PPA over subtasks
        if _task not in ppa_dict[_model]:
            mean_ppa = sum(ppa_values) / len(ppa_values) if ppa_values else None
            ppa_dict[_model][_task] = mean_ppa

df = pd.DataFrame.from_dict(ppa_dict, orient='index').style.apply(highlight_max_in_column, axis=0)

In [ ]:
# original_df is already your DataFrame
ppa_df = pd.DataFrame.from_dict(ppa_dict, orient='index')
mask = ~ppa_df.columns.astype(str).str.contains(r'-')
ppa_instruct_mean_df = ppa_df.loc[:, mask]

# inspect the result
ppa_instruct_mean_df

##### MRS

In [ ]:
mrs_dict = defaultdict(dict)
for _model in tqdm(ablation_models):
    mrs_values = []
    for _task in tasks:
        for _subtask in task_sub_task_mapping[_task]:
            _qrels_dict = load_qrels(_task, _subtask)
            _predictions = load_predictions(_model, _task, _subtask)
            _mrs_value = compute_mrs(_qrels_dict, _predictions)
            mrs_values.append(_mrs_value)
            mrs_dict[_model][_task + '-' + _subtask if _subtask is not None else _task] = _mrs_value
        
        # Compute mean MRS over subtasks
        if _task not in mrs_dict[_model]:
            mean_mrs = sum(mrs_values) / len(mrs_values) if mrs_values else None
            mrs_dict[_model][_task] = mean_mrs

df = pd.DataFrame.from_dict(mrs_dict, orient='index').style.apply(highlight_max_in_column, axis=0)

In [ ]:
mrs_df = pd.DataFrame.from_dict(mrs_dict, orient='index')
# original_df is already your DataFrame
mask = ~mrs_df.columns.astype(str).str.contains(r'-')
mrs_mean_df = mrs_df.loc[:, mask]

# inspect the result
mrs_mean_df